In [2]:
# Solution 1: Force PyTorch backend (avoids Keras issue)
import os
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'  # Reduce warnings

from transformers import pipeline
import gradio as gr
import torch

# Check if CUDA is available
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

# Initialize sentiment analysis pipeline with PyTorch backend
classifier = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    framework="pt",  # Force PyTorch framework
    device=device
)

def analyze_sentiment(text):
    if not text.strip():
        return "⚠️ Please enter some text to analyze!"
    
    try:
        # Get prediction from AI model
        result = classifier(text)[0]
        
        # Map labels to emojis
        emoji_map = {
            'LABEL_0': '😞 NEGATIVE',
            'LABEL_1': '😐 NEUTRAL', 
            'LABEL_2': '😊 POSITIVE',
            'NEGATIVE': '😞 NEGATIVE',
            'NEUTRAL': '😐 NEUTRAL',
            'POSITIVE': '😊 POSITIVE'
        }
        
        sentiment = emoji_map.get(result['label'], result['label'])
        confidence = result['score'] * 100
        
        return f"""
🤖 **AI Analysis Results:**
        
**Sentiment:** {sentiment}
**Confidence:** {confidence:.1f}%

💡 **Business Impact:**
{'🚨 Priority response needed!' if 'NEGATIVE' in sentiment else '✅ Customer satisfied!' if 'POSITIVE' in sentiment else '📝 Standard follow-up'}
        """
        
    except Exception as e:
        return f"❌ Error: {str(e)}\nTry installing: pip install torch"

# Create enhanced web interface
interface = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(
        placeholder="Enter customer feedback, review, or any text...",
        label="Customer Feedback",
        lines=3
    ),
    outputs=gr.Markdown(label="AI Analysis"),
    title="🤖 Real-time Customer Sentiment Analyzer",
    description="Powered by RoBERTa AI model - Analyze customer sentiment instantly!",
    examples=[
        ["This product is amazing! Great quality and fast delivery."],
        ["Terrible experience. Product broke after one day."],
        ["The product is okay, nothing special but does the job."],
        ["Outstanding service! Will definitely buy again."]
    ],
    theme=gr.themes.Soft()
)

if __name__ == "__main__":
    # Launch the app
    interface.launch(
        share=True,  # Creates public link
        server_name="0.0.0.0",
        server_port=7860
    )

Using device: CPU


pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cpu


Running on local URL:  http://0.0.0.0:7860
IMPORTANT: You are using gradio version 3.41.2, however version 4.44.1 is available, please upgrade.
--------

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
